<!-- STATUS_BLOCK_v1 -->
# indicator_validation.ipynb — STATUS

**Pipeline position:** Pipeline [3] — indicator QC on baseline only

**Purpose.**  Validate the 6 indicators on the treatment-free baseline window (4/13-4/22). For each indicator: coverage, noise floor (CV), within-baseline drift, split-half reliability, outlier days. PASS/WARN/FAIL per indicator.

**Inputs:**  `data/multi_day/{daily_summary, per_track_indicators, flower_visit_summary}.csv`
**Outputs:** verdict table inline; `data/multi_day/baseline_ranges.csv` (mean ± 2 SD per indicator)

### WORKS
- All 6 indicators validated (5/6 PASS, mean_handling_time_s has split-half WARN)
- Pre-registered baseline-range CSV written for the model

### PENDING
- Re-run once 5/10-5/12 are aggregated through multi_day_pipeline (already in the new run)

## Pipeline flow (canonical)

```
data/flight_data/<date>_system_<sys>/                  ← raw PATS-C output
       │
       ▼
[1] flower_visit_pipeline.ipynb                        slow; run when raw data changes
       └── data/multi_day/flower_visit_summary.csv

[2] multi_day_pipeline.ipynb                           always run after raw data updates
       ├── data/multi_day/daily_summary.csv
       ├── data/multi_day/per_track_indicators.csv
       └── data/multi_day/indicators_daily.csv         ← the file the model consumes

       (uses outputs of [1] + dBm + data transfer)
       │
       ▼
[3] validation.ipynb                                   sensor-integrity QC, run anytime
[3] indicator_validation.ipynb                         baseline-only QC of the 6 indicators
       │
       ▼
[4] exposure_analysis.ipynb                            figures + exploratory plots
[4] 5g_foraging_effect_model.ipynb                     pre-registered verdict, FINAL output
       │
       ▼
[5] statistical_methods.ipynb                          reading guide for [4]; not data-dependent
       │
       ▼
   paper / report
```

**Used in final report:**
- `5g_foraging_effect_model.ipynb` (verdict, mixed-effects model, composite FII)
- `exposure_analysis.ipynb` (figures)
- `validation.ipynb` (Methods)
- `indicator_validation.ipynb` (Methods)
- `statistical_methods.ipynb` (reference)

---


# Indicator validation (BASELINE only)

Every indicator we plan to use in `5g_foraging_effect_model.ipynb` is
validated here against the treatment-free baseline window (2026-04-13 to
2026-04-22).  The point: each indicator has to behave sensibly **before**
we apply 5G, otherwise any signal during ON/OFF is uninterpretable.

### The 6 indicators

| # | indicator | what it measures | direction (positive = 5G impairs) |
|---|---|---|---|
| 1 | `neg_exit_count`       | -1 × (n_v3 hive exits per day) | fewer trips → impairment |
| 2 | `neg_re_ratio`         | -1 × (returns / v3 exits) | unbalanced exit/return → impairment |
| 3 | `path_tortuosity`      | daily median tortuosity across all tracks | rougher paths → impairment |
| 4 | `ifi_cv`               | CV of inter-foraging interval (IFI) per day | more erratic timing → impairment |
| 5 | `mean_handling_time_s` | mean flower-visit duration (from flower-visit detector) | longer = less efficient extraction |
| 6 | `n_distinct_flowers`   | distinct spatial clusters visited per day | narrower foraging breadth → impairment |

### Validation checks

| check | rule | failing it means |
|---|---|---|
| V1 — coverage | ≥ 5 of 10 baseline days have non-NaN value | indicator might be unusable |
| V2 — noise floor | CV < 1.0 | day-to-day noise too large for the indicator to detect a real effect |
| V3 — drift | linear regression slope p-value > 0.05 | baseline isn't stable; need to detrend before treatment comparison |
| V4 — split-half | first-5 vs last-5 medians differ by < 30% relative | indicator isn't reliable across the baseline window |
| V5 — outlier-robust | no single day > 2 SD from mean | a single day dominates the baseline distribution |

A PASS on V1+V2 with V3-V5 informational is the minimum acceptance.
A WARN on V3 means the baseline is *drifting*, and the experimental
analysis should add a date covariate.


## Setup


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

DATA = Path("data/multi_day")
CYCLE_ANCHOR = pd.Timestamp("2026-04-23")
SYSTEM = 900   # sys 939 has only 2 baseline days; focus on sys 900

daily = pd.read_csv(DATA / "daily_summary.csv", parse_dates=["date"])
ptracks = pd.read_csv(DATA / "per_track_indicators.csv", parse_dates=["date","ts"])
fv = pd.read_csv(DATA / "flower_visit_summary.csv", parse_dates=["date"])

print(f"daily: {len(daily)} rows; per_track: {len(ptracks):,} rows; flower_visit_summary: {len(fv)} rows")


## Build the baseline indicator table


In [ ]:
# Filter to BASELINE only and sys 900 only
def is_baseline(d):
    return pd.Timestamp(d) < CYCLE_ANCHOR

daily_b   = daily[daily["date"].apply(is_baseline) & (daily["system_id"] == SYSTEM)].copy()
ptracks_b = ptracks[ptracks["date"].apply(is_baseline) & (ptracks["system_id"] == SYSTEM)].copy()
fv_b      = fv[fv["date"].apply(is_baseline) & (fv["system_id"] == SYSTEM)].copy()

# Indicators 1 & 2 from daily_summary
ind = daily_b[["date","n_v3","re_ratio_v3"]].copy()
ind["neg_exit_count"] = -ind["n_v3"]
ind["neg_re_ratio"]   = -ind["re_ratio_v3"]
ind = ind.drop(columns=["n_v3","re_ratio_v3"])

# Indicator 3: daily median tortuosity across all tracks (sys 900, baseline)
tort = ptracks_b.groupby("date")["tortuosity"].median().reset_index(name="path_tortuosity")
ind = ind.merge(tort, on="date", how="left")

# Indicator 4: IFI coefficient of variation per day
# (inter-foraging interval = time between consecutive v3 exits at the hive,
# CV = std / mean within the day)
def ifi_cv(group):
    ts = group.loc[group["hive_exit_v3"], "ts"].sort_values()
    if len(ts) < 3:
        return np.nan
    iv = ts.diff().dropna().dt.total_seconds()
    if iv.mean() == 0:
        return np.nan
    return iv.std() / iv.mean()

ifi = (ptracks_b.groupby("date").apply(ifi_cv).reset_index(name="ifi_cv"))
ind = ind.merge(ifi, on="date", how="left")

# Indicators 5 & 6 from flower_visit_summary
ind = ind.merge(fv_b[["date","mean_handling_time_s","n_distinct_flowers"]],
                 on="date", how="left")

ind = ind.sort_values("date").reset_index(drop=True)
INDICATORS = ["neg_exit_count","neg_re_ratio","path_tortuosity",
              "ifi_cv","mean_handling_time_s","n_distinct_flowers"]
print(f"Baseline days for system {SYSTEM}: {len(ind)}")
print()
print(ind.assign(date=lambda d: d["date"].dt.date).to_string(index=False))


## V1 — Coverage

Each indicator needs ≥ 5 non-NaN values out of the available baseline days
to be usable.


In [ ]:
coverage_rows = []
for ind_name in INDICATORS:
    n_total = len(ind)
    n_valid = ind[ind_name].notna().sum()
    coverage = n_valid / n_total
    verdict = "PASS" if n_valid >= 5 else "FAIL"
    coverage_rows.append({
        "indicator": ind_name, "n_valid": n_valid, "n_total": n_total,
        "coverage_pct": 100 * coverage, "verdict": verdict,
    })
cov_df = pd.DataFrame(coverage_rows)
print("=== V1: COVERAGE ===")
print(cov_df.to_string(index=False))


## V2 — Noise floor (Coefficient of Variation)

For each indicator: CV = SD / |mean| over the baseline.  A CV < 1 means the
day-to-day variation is smaller than the central tendency, leaving room
for treatment effects to be detected.  CV > 1.5 is alarming — the
indicator is too noisy to reliably pick up shifts of typical effect size.


In [ ]:
dist_rows = []
for ind_name in INDICATORS:
    s = ind[ind_name].dropna()
    if len(s) == 0:
        continue
    mean, sd, med = float(s.mean()), float(s.std()), float(s.median())
    iqr = float(s.quantile(0.75) - s.quantile(0.25))
    cv  = sd / abs(mean) if mean != 0 else np.nan
    if pd.isna(cv): verdict_cv = "N/A"
    elif cv < 0.5: verdict_cv = "LOW noise"
    elif cv < 1.0: verdict_cv = "MODERATE noise"
    elif cv < 1.5: verdict_cv = "HIGH noise (WARN)"
    else:          verdict_cv = "VERY HIGH noise (FAIL)"
    dist_rows.append({
        "indicator": ind_name, "n": len(s), "mean": mean, "median": med,
        "sd": sd, "IQR": iqr, "min": s.min(), "max": s.max(),
        "CV": cv, "verdict": verdict_cv,
    })
dist_df = pd.DataFrame(dist_rows)
print("=== V2: NOISE FLOOR ===")
print(dist_df.round(3).to_string(index=False))


## V3 — Within-baseline drift

A linear regression of indicator value vs day-index across the baseline.
If the slope is significantly different from 0 (`p_slope < 0.05`) the
baseline is *drifting*, which means the experimental analysis must either
exclude the early-baseline days or include date as a covariate.


In [ ]:
def linreg_slope(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = ~np.isnan(y); x = x[mask]; y = y[mask]
    n = len(x)
    if n < 3: return np.nan, np.nan, np.nan
    xm, ym = x.mean(), y.mean()
    sxx = ((x-xm)**2).sum()
    sxy = ((x-xm)*(y-ym)).sum()
    if sxx == 0: return np.nan, np.nan, np.nan
    slope = sxy / sxx
    intercept = ym - slope * xm
    yhat = slope * x + intercept
    resid = y - yhat
    s2 = (resid**2).sum() / (n - 2) if n > 2 else np.nan
    se_slope = np.sqrt(s2 / sxx) if not np.isnan(s2) else np.nan
    t = slope / se_slope if se_slope and se_slope > 0 else np.nan
    # crude two-sided p via normal approx (n=10, n-2=8, t-dist would be tighter)
    from math import erf, sqrt
    p = 2 * (1 - 0.5 * (1 + erf(abs(t) / sqrt(2)))) if not np.isnan(t) else np.nan
    return slope, p, t

drift_rows = []
for ind_name in INDICATORS:
    s = ind[ind_name]
    days = np.arange(len(ind))
    slope, p, t = linreg_slope(days, s.values)
    pct_of_mean = slope / abs(s.dropna().mean()) * 100 if s.dropna().mean() else np.nan
    drift_rows.append({
        "indicator": ind_name,
        "slope/day": slope, "slope%_of_mean": pct_of_mean,
        "t": t, "p": p,
        "verdict": ("STABLE" if pd.isna(p) or p >= 0.05 else "DRIFTING (WARN)"),
    })
drift_df = pd.DataFrame(drift_rows)
print("=== V3: BASELINE DRIFT ===")
print(drift_df.round(3).to_string(index=False))


## V4 — Split-half reliability

Split the baseline days into first half and second half, compute the
median per half, then look at the relative difference.  If the two halves
disagree by more than 30 % of the overall median, the indicator isn't
reliable across the baseline window.


In [ ]:
split_rows = []
n_days = len(ind)
half = n_days // 2
for ind_name in INDICATORS:
    s = ind[ind_name]
    first  = s.iloc[:half].dropna()
    second = s.iloc[half:].dropna()
    if len(first) < 2 or len(second) < 2:
        split_rows.append({"indicator": ind_name, "first_med": np.nan,
                            "second_med": np.nan, "rel_diff_pct": np.nan,
                            "verdict": "INSUFFICIENT DATA"})
        continue
    m1, m2 = float(first.median()), float(second.median())
    overall_med = float(s.dropna().median())
    rel = abs(m2 - m1) / abs(overall_med) * 100 if overall_med != 0 else np.nan
    verdict = "RELIABLE" if (pd.notna(rel) and rel < 30) else "DRIFTS BETWEEN HALVES (WARN)"
    split_rows.append({"indicator": ind_name, "first_med": m1,
                        "second_med": m2, "rel_diff_pct": rel, "verdict": verdict})
split_df = pd.DataFrame(split_rows)
print("=== V4: SPLIT-HALF RELIABILITY ===")
print(split_df.round(3).to_string(index=False))


## V5 — Outlier days

A day flagged as an outlier (> 2 SD from baseline mean) for ≥ 2 indicators
is treated as a candidate data-quality issue (probably a download problem
or a sensor outage), not a real biological signal.


In [ ]:
outlier_grid = pd.DataFrame(False, index=ind["date"].dt.date, columns=INDICATORS)
for ind_name in INDICATORS:
    s = ind[ind_name]
    m, sd = s.mean(), s.std()
    if sd and sd > 0:
        outlier_grid[ind_name] = (s.values > m + 2*sd) | (s.values < m - 2*sd)
outlier_grid["n_outlier_inds"] = outlier_grid.sum(axis=1)
print("=== V5: OUTLIER GRID (True = day > 2 SD from baseline mean for that indicator) ===")
print(outlier_grid.to_string())
print()
bad_days = outlier_grid[outlier_grid["n_outlier_inds"] >= 2].index.tolist()
print(f"Days flagged on >=2 indicators (candidate exclusions): {bad_days}")


## V6 — Inter-indicator correlation

If two indicators correlate at |ρ| > 0.9 across the baseline, they're
carrying nearly identical information and one is redundant.  We want the
6 indicators to span distinct dimensions of foraging behaviour.


In [ ]:
corr = ind[INDICATORS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(INDICATORS))); ax.set_xticklabels(INDICATORS, rotation=45, ha="right")
ax.set_yticks(range(len(INDICATORS))); ax.set_yticklabels(INDICATORS)
for i in range(len(INDICATORS)):
    for j in range(len(INDICATORS)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:+.2f}", ha="center", va="center",
                 color="white" if abs(v) > 0.5 else "black", fontsize=9)
ax.set_title("Spearman correlation across baseline (sys 900)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print()
print("Pairs with |rho| > 0.9 (redundant):")
for i in range(len(INDICATORS)):
    for j in range(i+1, len(INDICATORS)):
        if abs(corr.iloc[i, j]) > 0.9:
            print(f"  {INDICATORS[i]:<24s} vs {INDICATORS[j]:<24s}: rho = {corr.iloc[i,j]:+.3f}")


## Time series per indicator

Each indicator across the 10 baseline days, with mean ± 1 SD bands.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
days = ind["date"]
for ax, ind_name in zip(axes.flat, INDICATORS):
    s = ind[ind_name]
    ax.plot(days, s, marker="o", linewidth=1.5, color="#5B8FD4")
    m, sd = s.mean(), s.std()
    ax.axhline(m, color="black", linestyle="--", linewidth=0.8, label=f"mean = {m:.2f}")
    if sd and sd > 0:
        ax.fill_between(days, m - sd, m + sd, alpha=0.15, color="grey", label="±1 SD")
    ax.set_title(ind_name)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.legend(fontsize=8)
plt.suptitle(f"Baseline behaviour of each indicator (sys {SYSTEM}, {len(ind)} days)",
             fontweight="bold", y=1.0)
plt.tight_layout(); plt.show()


## Combined verdict per indicator

Aggregating V1-V5 into a single PASS/WARN/FAIL flag per indicator.


In [ ]:
verdict_table = (cov_df.set_index("indicator")["verdict"].rename("V1_coverage")
                  .to_frame()
                  .join(dist_df.set_index("indicator")["verdict"].rename("V2_noise"))
                  .join(drift_df.set_index("indicator")["verdict"].rename("V3_drift"))
                  .join(split_df.set_index("indicator")["verdict"].rename("V4_split"))
                 )
# Outlier impact: did this indicator contribute to >=2 of the flagged bad days?
def v5_for(ind_name):
    bad = outlier_grid.loc[outlier_grid["n_outlier_inds"] >= 2, ind_name]
    return "OUTLIERS PRESENT" if bad.any() else "CLEAN"
verdict_table["V5_outliers"] = [v5_for(i) for i in verdict_table.index]

def overall(row):
    if any("FAIL" in v for v in row): return "FAIL"
    if any("WARN" in v for v in row): return "WARN"
    return "PASS"
verdict_table["OVERALL"] = verdict_table.apply(overall, axis=1)

print("=== INDICATOR-VALIDATION SUMMARY ===")
print(verdict_table.to_string())


## Pre-registered baseline ranges

For each indicator, the (mean - 2 SD, mean + 2 SD) window from the baseline
gives a "normal range" that experimental-period (OFF) days should sit
inside if the colony is behaving the same way it did pre-experiment.
If experimental OFF days fall outside this window, that's a sign the
colony's baseline behaviour has shifted (acclimatisation, seasonal change,
or experimenter effects unrelated to 5G).

Paste this table into the Methods section.


In [ ]:
prereg_rows = []
for ind_name in INDICATORS:
    s = ind[ind_name].dropna()
    if len(s) < 3:
        prereg_rows.append({"indicator": ind_name, "baseline_mean": np.nan,
                             "baseline_sd": np.nan, "expected_lo": np.nan,
                             "expected_hi": np.nan})
        continue
    m, sd = float(s.mean()), float(s.std())
    prereg_rows.append({"indicator": ind_name, "baseline_mean": m, "baseline_sd": sd,
                         "expected_lo": m - 2*sd, "expected_hi": m + 2*sd})
prereg_df = pd.DataFrame(prereg_rows)
print("=== PRE-REGISTERED BASELINE RANGES (mean +/- 2 SD) ===")
print(prereg_df.round(3).to_string(index=False))
print()
print("Save this to data/multi_day/baseline_ranges.csv for the model to consume.")
prereg_df.to_csv(DATA / "baseline_ranges.csv", index=False)
print(f"Wrote {DATA / 'baseline_ranges.csv'}")
